# Preproducción — Lead Scoring


In [1]:
import json
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split


In [2]:
NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
TARGET = 'compra'
ID_COLUMN = 'id'
csv_path = PROJECT_ROOT / '02_datos' / '01_Originales' / 'Leads.csv'
preprocessor_path = PROJECT_ROOT / '05_modelos' / 'preprocesador.joblib'
finalists_path = PROJECT_ROOT / '01_Documentos' / 'Variables_preseleccionadas.txt'
model_config_path = PROJECT_ROOT / '06_resultados' / 'Modelizacion' / 'config_mejor_modelo.json'


In [3]:
required_paths = [csv_path, preprocessor_path, finalists_path, model_config_path]
missing_paths = [path for path in required_paths if not path.exists()]
assert not missing_paths, f'Artefactos requeridos ausentes: {missing_paths}'
print(f'CSV: {csv_path}')
print(f'Preprocesador: {preprocessor_path}')


CSV: c:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\02_datos\01_Originales\Leads.csv
Preprocesador: c:\Users\Dell\Agus\Master Agentic DS\Lead_Scoring\05_modelos\preprocesador.joblib


In [4]:
leads = pd.read_csv(csv_path, sep=';', encoding='utf-8')
df_train, df_validation = train_test_split(leads, test_size=0.30, random_state=42)
df = df_train.copy()


In [5]:
for column in ['ocupacion', 'ambito', 'ult_actividad', 'fuente']:
    df[column] = df[column].fillna('Desconocido')

df['visitas_total_missing'] = df['visitas_total'].isna().astype('int8')
df['visitas_total'] = df['visitas_total'].fillna(3.0)
df['paginas_vistas_visita_missing'] = df['paginas_vistas_visita'].isna().astype('int8')
df['paginas_vistas_visita'] = df['paginas_vistas_visita'].fillna(2.0)
df['fuente'] = df['fuente'].replace({'google': 'Google'})

mask_extremos = (df['visitas_total'] > 30) | (df['paginas_vistas_visita'] > 20)
df = df.loc[~mask_extremos].copy()


In [6]:
fuente_principales = ['Google', 'Direct Traffic', 'Chat', 'Organic Search']
df['fuente'] = df['fuente'].where(df['fuente'].isin(fuente_principales), 'Otros')

umbral_ult_actividad = 0.02
freq_ult = df['ult_actividad'].value_counts(normalize=True, dropna=False)
categorias_ult_otros = freq_ult[freq_ult < umbral_ult_actividad].index.tolist()
df['ult_actividad'] = df['ult_actividad'].where(~df['ult_actividad'].isin(categorias_ult_otros), 'Otros')


In [7]:
X_raw = df.drop(columns=[TARGET, ID_COLUMN])
y_train = df[TARGET].copy()
preprocessor = joblib.load(preprocessor_path)
transformed_array = preprocessor.transform(X_raw)
feature_names = list(preprocessor.get_feature_names_out())
name_map = {
    'behavior__visitas_total': 'visitas_total_yj_mm',
    'behavior__tiempo_en_site_total': 'tiempo_en_site_total_yj_mm',
    'behavior__paginas_vistas_visita': 'paginas_vistas_visita_yj_mm',
    'score_values__score_actividad': 'score_actividad_mm',
    'score_values__score_perfil': 'score_perfil_mm',
    'score_missing__missingindicator_score_actividad': 'score_actividad_missing',
}
final_feature_names = [name_map.get(name, name.replace('categorical__', '').replace('binary__', '')) for name in feature_names]
features = pd.DataFrame(transformed_array, index=df.index, columns=final_feature_names)


c:\Users\Dell\miniconda3\envs\nivii_ai\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator PowerTransformer from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Dell\miniconda3\envs\nivii_ai\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator MinMaxScaler from version 1.9.1 when using version 1.9.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Dell\miniconda3\envs\nivii_ai\Lib\site-packages\sklearn\base.py:525: InconsistentVersionWarning: Trying to unpickle estimator Pipeline from version 1.9.1 when using versio

In [ ]:
variables_finalistas = [line.strip() for line in finalists_path.read_text(encoding='utf-8').splitlines() if line.strip()]
missing_finalists = sorted(set(variables_finalistas) - set(features.columns))
assert not missing_finalists, f'Variables finalistas no producidas: {missing_finalists}'
X_train_final = features.loc[:, variables_finalistas].copy()
print(f'Filas de entrenamiento: {len(X_train_final)}')
print(f'Predictoras finalistas: {X_train_final.shape[1]}')


In [ ]:
model_config = json.loads(model_config_path.read_text(encoding='utf-8'))
model_params = model_config['parametros']
final_model = LogisticRegression(
    C=float(model_params['C']),
    penalty=model_params['penalty'],
    solver=model_params['solver'],
    max_iter=int(model_params['max_iter']),
    random_state=int(model_params['random_state']),
)
final_model.fit(X_train_final, y_train)
